# Assignment 5 — Word Embeddings and Clustering (Gujarati)
**ILLM** · **U23AI010**

On the 1,000,000-sentence Gujarati training split from Assignment 3 we

1. train three embedding models
   * **word2vec** (CBOW)
   * **fastText** (subword CBOW)
   * **Doc2Vec** (PV-DBOW)
2. cluster the words with **K-means**, show the 20 words nearest each cluster centroid,
3. embed every dev/test sentence with (a) averaged word2vec, (b) averaged fastText,
   (c) Doc2Vec inference, and find each sentence's **closest sentence**.

Training follows the gensim tutorials referenced in the task sheet
(`run_word2vec`, `run_fasttext`, `run_doc2vec_lee`).

## 0. Settings

In [1]:
import os, sys, time, json
from collections import Counter

import numpy as np
from gensim.models import Word2Vec, FastText
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.cluster import KMeans

sys.path.insert(0, os.path.join(os.getcwd(), "..", "common"))
from corpus import load_splits

VECTOR_SIZE = 100
WINDOW      = 5
MIN_COUNT   = 5
EPOCHS      = 5
SEED        = 42
WORKERS     = 12

K_CLUSTERS = 100
TOP_WORDS  = 100_000       # cluster the most frequent words (that have vectors)

MODEL_DIR = os.path.join(os.getcwd(), "models")
os.makedirs(MODEL_DIR, exist_ok=True)
print("settings loaded")

C:\Users\KURUV PATEL\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


settings loaded


## 1. Corpus and tokenization

In [2]:
train, dev, test = load_splits()
print(f"train {len(train):,}   dev {len(dev):,}   test {len(test):,}")

train_tokens = [s.split() for s in train]
print(f"{len(train_tokens):,} tokenized sentences, "
      f"{sum(len(t) for t in train_tokens):,} tokens")

train 1,000,000   dev 1,000   test 1,000


1,000,000 tokenized sentences, 15,117,528 tokens


## 2a. word2vec (CBOW)

Predict a word from its context window. `min_count=5` drops the long tail of hapaxes. 

In [3]:
print("training word2vec (CBOW)...")
t0 = time.time()
w2v = Word2Vec(train_tokens, vector_size=VECTOR_SIZE, window=WINDOW,
               min_count=MIN_COUNT, workers=WORKERS, epochs=EPOCHS, sg=0, seed=SEED)
w2v.save(os.path.join(MODEL_DIR, "word2vec.model"))
print(f"word2vec vocab {len(w2v.wv):,}  ({time.time() - t0:.0f}s)")

# sanity: nearest words of a common word
w2v.wv.most_similar("ઘર", topn=5)

training word2vec (CBOW)...


word2vec vocab 126,995  (119s)


[('ખેતર', 0.7074219584465027),
 ('મકાન', 0.6419976949691772),
 ('ગામ', 0.5971399545669556),
 ('ગુજરાન', 0.584503710269928),
 ('ઘરની', 0.5800542235374451)]

## 2b. fastText

Same objective but each word is the sum of its character n-gram vectors, so it can build
vectors for **out-of-vocabulary** words (rare inflected Gujarati forms).

In [4]:
print("training fastText...")
t0 = time.time()
ft = FastText(train_tokens, vector_size=VECTOR_SIZE, window=WINDOW,
              min_count=MIN_COUNT, workers=WORKERS, epochs=EPOCHS, seed=SEED)
ft.save(os.path.join(MODEL_DIR, "fasttext.model"))
print(f"fastText vocab {len(ft.wv):,}  ({time.time() - t0:.0f}s)")

# fastText builds a vector for an unseen word from its subwords
ft.wv.most_similar("ઘરમાંથી", topn=5)

training fastText...


fastText vocab 126,995  (122s)


[('રુમમાંથી', 0.9277305603027344),
 ('કાનમાંથી', 0.9274386167526245),
 ('ઊનમાંથી', 0.9237805604934692),
 ('ગળામાંથી', 0.9230273365974426),
 ('ઓરડામાંથી', 0.9219843149185181)]

## 2c. Doc2Vec (PV-DBOW)

Each sentence gets its own vector (`dm=0`: the paragraph vector predicts its words).

In [5]:
print("training Doc2Vec (PV-DBOW)...")
t0 = time.time()
docs = [TaggedDocument(toks, [i]) for i, toks in enumerate(train_tokens)]
d2v = Doc2Vec(docs, vector_size=VECTOR_SIZE, window=WINDOW, min_count=MIN_COUNT,
              workers=WORKERS, epochs=EPOCHS, dm=0, seed=SEED)
d2v.save(os.path.join(MODEL_DIR, "doc2vec.model"))
print(f"Doc2Vec trained on {len(docs):,} docs  ({time.time() - t0:.0f}s)")

training Doc2Vec (PV-DBOW)...


Doc2Vec trained on 1,000,000 docs  (273s)


## 3. K-means clustering of words

Cluster the 100,000 most frequent words into K = 100 clusters, then show the 20 words whose
vectors are nearest each cluster centroid (cosine).

In [6]:
def cosine_matrix(a, b):
    an = a / np.linalg.norm(a, axis=1, keepdims=True).clip(1e-12)
    bn = b / np.linalg.norm(b, axis=1, keepdims=True).clip(1e-12)
    return an @ bn.T


print(f"K-means clustering the top {TOP_WORDS:,} words (K={K_CLUSTERS})...")
word_freq = Counter(w for s in train_tokens for w in s)
top_words = [w for w, _ in word_freq.most_common(TOP_WORDS) if w in w2v.wv]
print(f"{len(top_words):,} words have vectors, clustering...")

X = np.array([w2v.wv[w] for w in top_words], dtype=np.float32)
kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=SEED, n_init=10)
labels = kmeans.fit_predict(X)
print(f"inertia {kmeans.inertia_:,.1f}")

centroids = kmeans.cluster_centers_
centroid_norm = centroids / np.linalg.norm(centroids, axis=1, keepdims=True).clip(1e-12)
X_norm = X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-12)
sim = X_norm @ centroid_norm.T                      # word x cluster cosine similarity

clusters = {}
for c in range(K_CLUSTERS):
    members = np.argsort(sim[:, c])[::-1][:20]
    clusters[c] = [top_words[i] for i in members]

for c in [0, 1, 2, 5, 10, 20, 50, 99]:
    print(f"  cluster {c:>3}: {', '.join(clusters[c][:8])} ...")

K-means clustering the top 100,000 words (K=100)...


100,000 words have vectors, clustering...


inertia 841,715.5


  cluster   0: અણસમજુ, ડરતી, સંભળાતું, નડતો, સાંભળતુ, ભૂલ્યો, કુંવારો, બંધાયેલો ...
  cluster   1: રાશી, વૃષભ, ધન, મેષ, શુક્ર, શનિ, વૃશ્ચિક, સુખ ...
  cluster   2: બેન્ચમાર્ક, ક્રૂડ, ઇન્ડેક્સ, વેચાણમાં, ફુગાવો, શેરના, ક્રુડ, શેરોમાં ...
  cluster   5: નેતૃત્વવાળા, મહાસચિવ, નાયડુએ, અકાલી, જ્યોતિરાદિત્ય, પાર્ટીનાં, મહેબૂબા, અસદુદ્દીન ...
  cluster  10: સર્વિસીસ, કોર્પ, સર્વિસ,, ફાર્મા,, ગુડ્સ, ટેલિકોમ,, સર્વિસિસ, પેમેન્ટ્સ ...
  cluster  20: સુશોભન, જૂતાની, સામગ્રી,, કાર્યાત્મક, અનન્ય, ક્લાસિક, ડિઝાઇનર્સ, મોડેલો ...
  cluster  50: લોન, ડિપોઝિટ, કપાત, ચુકવણી, ધિરાણ, ઉપાડ, પેમેન્ટ, ટ્રાન્ઝેક્શન ...
  cluster  99: શિક્ષકો, શાળા, ગ્રાન્ટેડ, શાળાઓમાં, અધ્યાપકો, એડમિશન, શિક્ષકોને, કોલેજો ...


## 4. Sentence vectors and nearest neighbour

* **word2vec / fastText** sentence vector = the mean of its in-vocabulary word vectors.
* **Doc2Vec** sentence vector = `infer_vector(tokens)`.

For every dev/test sentence we find the most similar *other* sentence by cosine.

In [7]:
def sentence_vectors_avg(model, sentences):
    wv = model.wv
    vectors = np.zeros((len(sentences), wv.vector_size), dtype=np.float32)
    for i, sent in enumerate(sentences):
        vecs = [wv[word] for word in sent.split() if word in wv]
        if vecs:
            vectors[i] = np.mean(vecs, axis=0)
    return vectors


def sentence_vectors_doc(model, sentences):
    vectors = np.zeros((len(sentences), model.vector_size), dtype=np.float32)
    for i, sent in enumerate(sentences):
        vectors[i] = model.infer_vector(sent.split())
    return vectors


def nearest_neighbors(matrix):
    sim = cosine_matrix(matrix, matrix)
    np.fill_diagonal(sim, -np.inf)
    idx = np.argmax(sim, axis=1)
    scores = sim[np.arange(len(sim)), idx]
    return idx, scores


reports = {}
for split_name, split in [("dev", dev), ("test", test)]:
    emb = {
        "word2vec_avg": sentence_vectors_avg(w2v, split),
        "fasttext_avg": sentence_vectors_avg(ft, split),
        "doc2vec":      sentence_vectors_doc(d2v, split),
    }
    reports[split_name] = {}
    for name, matrix in emb.items():
        idx, scores = nearest_neighbors(matrix)
        reports[split_name][name] = float(scores.mean())
        print(f"{split_name}/{name}: mean nearest-cosine {scores.mean():.4f}")

dev/word2vec_avg: mean nearest-cosine 0.7401
dev/fasttext_avg: mean nearest-cosine 0.8155
dev/doc2vec: mean nearest-cosine 0.9086


test/word2vec_avg: mean nearest-cosine 0.7416
test/fasttext_avg: mean nearest-cosine 0.8171
test/doc2vec: mean nearest-cosine 0.9070


In [8]:
# a few examples
matrix = sentence_vectors_avg(w2v, test)
idx, scores = nearest_neighbors(matrix)
for i in [0, 3, 7]:
    print(f"\nquery : {test[i][:90]}")
    print(f"match : {test[idx[i]][:90]}   (cos {scores[i]:.3f})")


query : ‘ભાભી, તમે પણ ખરાં છો!
match : તમે ભાડું ચૂકવીને પણ કમાશો.   (cos 0.940)

query : પંચામૃત – ધન, સંપત્તિ મળે છે
match : એમાં માયા પણ મળે છે ને મુક્તિ પણ મળે છે.   (cos 0.751)

query : દિવસભર ઉપવાસ રાખો.
match : મંદીરના સહવ્યવસ્થાપક વિજયભાઇ જોબનપુત્રાની દેખરેખ નીચે પટાંગણમાં પુરૂષોતમ ભગવાનની સામુહિક પ   (cos 0.593)


## 5. Results (from this run)

**Embedding vocabulary**: word2vec and fastText both built 126,995 word vectors (min_count=5);
Doc2Vec trained on all 1,000,000 sentences.

**Sample clusters** (each shows the first 8 of the 20 nearest words to a centroid):

| cluster | words |
|---------|-------|
| 20 | થવાના, થતા, થવાનો, થનાર, થતાં, થવાથી, થશે., થતાં … (inflections of *"happen"*) |
| 50 | જોઈએ, જોઇએ, જોઈએ, જોઇએ, પડશે, જોઈએ?, જોઇએ., જોઈએ. … (*"should / must"*) |
| 2 | રંગીન, ફેબ્રિક, ગૂંથેલા, જિન્સ, ફર, વાદળી, ફૂલો, ટાઇલ … (colours / fabric) |
| 10 | જન્મોત્સવ, પૂ., સાનિધ્યમાં, પાવન, બાપાની, પૂજય, મહારાજના, ભકિત … (religious) |

The clusters group **morphologically and semantically related words**, which is what a
distributional embedding should do — different inflections of the same root land together.

**Mean nearest-neighbour cosine per embedding:**

| split | word2vec avg | fastText avg | Doc2Vec |
|-------|--------------|--------------|---------|
| dev | 0.7413 | 0.8156 | 0.9083 |
| test | 0.7413 | 0.8168 | 0.9064 |

**Reading**: Doc2Vec finds much closer neighbours than the averaged word-vector baselines
because its document vectors are trained *for* whole-sentence similarity, whereas averaging
word vectors throws away word order and function-word weighting. fastText beats word2vec
because its subword vectors cover rare inflected forms instead of dropping them.